# ĐỒ ÁN CHUYÊN ĐỀ THIẾT KẾ ỨNG DỤNG THIẾT BỊ SỐ - MKU
## BUỔI 8: HUẤN LUYỆN MÔ HÌNH NHẬN DIỆN NGUYÊN LIỆU THỰC PHẨM (TENSORFLOW / KERAS -> TFLITE)
**Đề tài:** Ứng dụng gợi ý món ăn thông minh dựa trên nguyên liệu (SmartCook AI)

Notebook này được chuẩn bị để chạy trực tiếp trên **Google Colab (GPU)** hoặc **Kaggle** nhằm huấn luyện mô hình Transfer Learning (MobileNetV2) nhận diện các loại nguyên liệu nấu ăn phổ biến tại Việt Nam (Cà chua, Trứng, Thịt bò, Thịt gà, Hành tây, Cà rốt, Khoai tây, Nấm rơm, Tôm, Đậu phụ, Bắp cải, v.v.) và xuất ra tập tin `ingredient_classifier.tflite`.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

### 1. Cấu hình Siêu tham số & Kích thước ảnh (224x224 theo yêu cầu Buổi 7 & 8)

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
DATASET_DIR = './ingredient_dataset'  # Thư mục chứa các folder con theo tên nguyên liệu

### 2. Tải và Tiền xử lý dữ liệu (Data Augmentation)

In [ ]:
# Tạo dữ liệu huấn luyện và kiểm thử với tỉ lệ 80% - 20%
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("Danh sách nhãn nguyên liệu nhận diện:", class_names)

# Lưu nhãn vào file ingredient_labels.txt nạp vào Android Assets
with open('ingredient_labels.txt', 'w', encoding='utf-8') as f:
    for name in class_names:
        f.write(f"{name}\n")

In [ ]:
# Data Augmentation để tăng độ bền vững khi chụp trong nhiều góc độ / điều kiện ánh sáng
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

### 3. Xây dựng Kiến trúc Mô hình (MobileNetV2 Transfer Learning)

In [ ]:
# Khởi tạo backbone MobileNetV2 pretrained trên ImageNet, loại bỏ lớp phân loại đầu ra
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Đóng băng các tầng trích xuất đặc trưng

inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)  # Chuẩn hóa về [-1, 1]
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

### 4. Huấn luyện mô hình

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

### 5. Xuất mô hình sang định dạng TensorFlow Lite (.tflite)

In [ ]:
# Chuyển đổi Keras Model -> TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# Tối ưu hóa kích thước mô hình cho thiết bị di động
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = 'ingredient_classifier.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"Đã xuất thành công mô hình TFLite: {tflite_path}")
print(f"Kích thước tập tin: {len(tflite_model) / (1024 * 1024):.2f} MB")
print("-> Copy tập tin ingredient_classifier.tflite và ingredient_labels.txt vào thư mục app/src/main/assets của dự án Android!")